# DS-005: Advanced Models - Phase 1 (Gradient Boosting)

**Date**: 15/06/2025 19:13  
**Phase**: DS-005 Advanced Models - Phase 1  
**Objective**: Implement XGBoost and LightGBM to exceed baseline F1=43.58%  
**Target**: F1≥70% (minimum), F1≥90% (stretch goal)  

## 🎯 Current Baseline to Beat
- **Best Model**: Logistic Regression  
- **F1-Score**: 43.58%  
- **Precision**: 27.97%  
- **Recall**: 98.65%  

## 📋 Implementation Plan
1. **Data Loading & Analysis**
2. **XGBoost Implementation** - Scale_pos_weight optimization
3. **LightGBM Implementation** - Class_weight balancing
4. **Hyperparameter Optimization** - Bayesian optimization with Optuna
5. **Performance Analysis & Comparison**
6. **Model Selection & Validation**


## 1. Environment Setup & Data Loading


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Advanced ML libraries
import xgboost as xgb
import lightgbm as lgb
import optuna
from optuna import Trial

# Core ML libraries
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Utilities
import joblib
import json
import os
from pathlib import Path

print(f"🚀 DS-005 Advanced Models Started: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"📊 XGBoost version: {xgb.__version__}")
print(f"📊 LightGBM version: {lgb.__version__}")
print(f"📊 Optuna version: {optuna.__version__}")


In [ ]:
# Load processed data
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')
val_df = pd.read_csv('../data/processed/validation.csv')

print(f"📈 Training data shape: {train_df.shape}")
print(f"📈 Test data shape: {test_df.shape}")
print(f"📈 Validation data shape: {val_df.shape}")

# Check class distribution
print("\\n📊 Class Distribution in Training Data:")
print(train_df['label'].value_counts(normalize=True))

# Calculate class weights for imbalanced dataset (3:1 Ham:Spam)
class_counts_encoded = train_df['label_encoded'].value_counts()
total_samples = len(train_df)
class_weight_ratio = class_counts_encoded[0] / class_counts_encoded[1]  # ham(0)/spam(1) ratio
print(f"\\n⚖️ Class weight ratio (Ham:Spam): {class_weight_ratio:.2f}:1")
print(f"Label mapping: 0=Ham ({class_counts_encoded[0]}), 1=Spam ({class_counts_encoded[1]})")


In [ ]:
# Load the TF-IDF vectorizer from baseline models
tfidf_vectorizer = joblib.load('../models/tfidf_vectorizer_v1.0.0.joblib')
print(f"📊 TF-IDF vectorizer loaded: {tfidf_vectorizer.get_feature_names_out().shape[0]} features")

# Check available columns and use the appropriate text column
print(f"\\n📋 Available columns: {train_df.columns.tolist()}")

# Use text_aggressive as it likely contains the most processed version for ML
text_column = 'text_aggressive'
print(f"\\n📝 Using text column: '{text_column}'")

# Transform text data to TF-IDF features
X_train = tfidf_vectorizer.transform(train_df[text_column])
X_test = tfidf_vectorizer.transform(test_df[text_column])
X_val = tfidf_vectorizer.transform(val_df[text_column])

y_train = train_df['label_encoded']
y_test = test_df['label_encoded']
y_val = val_df['label_encoded']

print(f"\\n✅ Features prepared: {X_train.shape[1]} TF-IDF features")
print(f"✅ Training samples: {X_train.shape[0]}")
print(f"✅ Test samples: {X_test.shape[0]}")
print(f"✅ Validation samples: {X_val.shape[0]}")


## 3. XGBoost Implementation with Class Balance Optimization


In [ ]:
# XGBoost baseline model with scale_pos_weight optimization
print("🚀 Starting XGBoost Implementation...")

# Initial XGBoost model with class imbalance handling
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=class_weight_ratio,  # Handle 3:1 class imbalance
    random_state=42,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

# Train the model
print("🔄 Training XGBoost baseline model...")
xgb_model.fit(X_train, y_train)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Calculate metrics
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"\\n📊 XGBoost Baseline Results:")
print(f"F1-Score: {xgb_f1:.4f} ({xgb_f1*100:.2f}%)")
print(f"Precision: {xgb_precision:.4f} ({xgb_precision*100:.2f}%)")
print(f"Recall: {xgb_recall:.4f} ({xgb_recall*100:.2f}%)")
print(f"AUC-ROC: {xgb_auc:.4f}")

# Compare with baseline
baseline_f1 = 0.4358
improvement = ((xgb_f1 - baseline_f1) / baseline_f1) * 100
print(f"\\n🎯 Improvement over baseline: {improvement:+.2f}%")
print(f"Target achieved: {'✅ YES' if xgb_f1 >= 0.70 else '⚠️ PARTIAL' if xgb_f1 > baseline_f1 else '❌ NO'}")


In [ ]:
# LightGBM implementation with class weight handling
print("🚀 Starting LightGBM Implementation...")

# Calculate class weights for LightGBM
n_pos = sum(y_train == 1)
n_neg = sum(y_train == 0)
lgb_class_weight = {0: 1.0, 1: n_neg/n_pos}

# Initial LightGBM model
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    boosting_type='gbdt',
    class_weight=lgb_class_weight,
    random_state=42,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_jobs=-1,
    verbose=-1
)

# Train the model
print("🔄 Training LightGBM baseline model...")
lgb_model.fit(X_train, y_train)

# Make predictions
y_pred_lgb = lgb_model.predict(X_test)
y_pred_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Calculate metrics
lgb_f1 = f1_score(y_test, y_pred_lgb)
lgb_precision = precision_score(y_test, y_pred_lgb)
lgb_recall = recall_score(y_test, y_pred_lgb)
lgb_auc = roc_auc_score(y_test, y_pred_proba_lgb)

print(f"\\n📊 LightGBM Baseline Results:")
print(f"F1-Score: {lgb_f1:.4f} ({lgb_f1*100:.2f}%)")
print(f"Precision: {lgb_precision:.4f} ({lgb_precision*100:.2f}%)")
print(f"Recall: {lgb_recall:.4f} ({lgb_recall*100:.2f}%)")
print(f"AUC-ROC: {lgb_auc:.4f}")

# Compare with baseline
lgb_improvement = ((lgb_f1 - baseline_f1) / baseline_f1) * 100
print(f"\\n🎯 Improvement over baseline: {lgb_improvement:+.2f}%")
print(f"Target achieved: {'✅ YES' if lgb_f1 >= 0.70 else '⚠️ PARTIAL' if lgb_f1 > baseline_f1 else '❌ NO'}")


In [ ]:
# Create comprehensive comparison
results_df = pd.DataFrame({
    'Model': ['Logistic Regression (Baseline)', 'XGBoost', 'LightGBM'],
    'F1-Score': [0.4358, xgb_f1, lgb_f1],
    'Precision': [0.2797, xgb_precision, lgb_precision],
    'Recall': [0.9865, xgb_recall, lgb_recall],
    'AUC-ROC': [np.nan, xgb_auc, lgb_auc]
})

# Calculate improvements
results_df['F1_Improvement_%'] = (
    (results_df['F1-Score'] - results_df['F1-Score'].iloc[0]) / 
    results_df['F1-Score'].iloc[0] * 100
)

print("📊 COMPREHENSIVE MODEL COMPARISON")
print("=" * 80)
print(results_df.round(4))

# Identify best model
best_model_idx = results_df['F1-Score'].iloc[1:].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1 = results_df.loc[best_model_idx, 'F1-Score']

print(f"\\n🏆 BEST PERFORMING MODEL: {best_model_name}")
print(f"🎯 Best F1-Score: {best_f1:.4f} ({best_f1*100:.2f}%)")

# Target achievement analysis
print("\\n🎯 TARGET ACHIEVEMENT ANALYSIS:")
print(f"Minimum Target (F1≥70%): {'✅ ACHIEVED' if best_f1 >= 0.70 else '❌ NOT YET ACHIEVED'}")
print(f"Stretch Target (F1≥90%): {'✅ ACHIEVED' if best_f1 >= 0.90 else '❌ NOT YET ACHIEVED'}")
print(f"Gap to minimum target: {max(0, (0.70 - best_f1)*100):.2f} percentage points")
print(f"Gap to stretch target: {max(0, (0.90 - best_f1)*100):.2f} percentage points")

# Save initial results
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
results_filename = f'../models/advanced_models_initial_results_{timestamp}.csv'
results_df.to_csv(results_filename, index=False)
print(f"\\n💾 Results saved to: {results_filename}")

print("\\n🔄 NEXT STEPS:")
print("1. Hyperparameter optimization with Optuna")
print("2. Advanced feature engineering")
print("3. Ensemble methods if needed")
print("4. Cross-validation analysis")
